# Multiprocess multiple day folders and summarize the flux timeseries (MCMC)

Point `parent_data_folder` at a directory that contains **one subfolder per day** (e.g. `.../1-1/data/`, with children like `2026-04-02/`, `2026-04-03/`, ...).

For each day folder, this notebook runs the `Multiprocessor` in MCMC mode, selects the Pareto-optimal `(deadband, cutoff)` per measurement, writes one `*_bestPareto.nc` file and one summary CSV per folder into `output_folder`, and plots the resulting `dcdt(HM)` timeseries with a 16–84 percentile band.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

In [ ]:
import logging
import pathlib

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

from soilgasflux_fcs import Multiprocessor, json_reader

logging.getLogger('soilgasflux_fcs').setLevel(logging.INFO)

In [ ]:
# parent_data_folder = pathlib.Path('/Users/alexnaokiasatokobayashi/Documents/data/Kerzers/CHYN-Kerzers01/low-cost_sensor/kerzers_20260414/2-1/data/')
parent_data_folder = pathlib.Path('/Users/alexnaokiasatokobayashi/Documents/data/Gals/gals_20250829/sgf_07_pt02/data')
# '/Users/alexnaokiasatokobayashi/Documents/data/Gals/2026_tpclass/low-cost sensor/2-2/data/2026-03-19'
output_folder = pathlib.Path('/Users/alexnaokiasatokobayashi/Downloads/test_output_gals')
output_folder.mkdir(parents=True, exist_ok=True)

# `chamber_id` is taken from the data-folder ancestor (same convention as multiprocess_folder_summary.ipynb).
chamber_id = parent_data_folder.parent.name

# Discover day folders: each child directory of `parent_data_folder` is treated as one day of data.
day_folders = sorted(p for p in parent_data_folder.iterdir() if p.is_dir())
print(f'chamber_id = {chamber_id}')
print(f'{len(day_folders)} day folders found:')
for p in day_folders:
    print(f'  - {p.name}')

For each day folder: load the raw data, run `Multiprocessor.run_MC`, then `select_bestPareto` writes `<chamber_id>_<date>_bestPareto.nc` into `output_folder`. A per-folder summary CSV is also saved alongside the netCDF.

In [ ]:
# --- per-day logging setup ----------------------------------------------------
# One log file per day folder. The FileHandler on `pkg_logger` is swapped at the
# start of each iteration, and the same path is handed to run_MC(log_path=...)
# so the pool's workers write into the same per-day file.
pkg_logger = logging.getLogger('soilgasflux_fcs')
pkg_logger.setLevel(logging.INFO)
pkg_logger.propagate = False  # do not bubble to root (stderr / notebook)
nb_log = logging.getLogger('soilgasflux_fcs.notebook')

_log_fmt = logging.Formatter('%(asctime)s %(levelname)-7s %(name)s: %(message)s')


def _attach_day_log(day_name):
    """Point pkg_logger at a fresh per-day log file, return its path."""
    day_log = output_folder / f'{chamber_id}_{day_name}_processing.log'
    for h in list(pkg_logger.handlers):
        if isinstance(h, logging.FileHandler):
            pkg_logger.removeHandler(h)
            h.close()
        elif isinstance(h, logging.StreamHandler):
            # defensive: strip any stray stderr handler
            pkg_logger.removeHandler(h)
    fh = logging.FileHandler(day_log, mode='a')
    fh.setFormatter(_log_fmt)
    pkg_logger.addHandler(fh)
    return day_log


processor = Multiprocessor()
summaries = {}
log_paths = {}


def _build_summary(pareto_files):
    ds_best = xr.open_mfdataset(pareto_files, combine='by_coords').sortby('time')
    dcdt = ds_best['dcdt(HM)']
    return pd.DataFrame({
        'time': ds_best['time'].values,
        'dcdt_median': dcdt.median(dim='MC').values,
        'dcdt_q16': dcdt.quantile(0.16, dim='MC').values,
        'dcdt_q84': dcdt.quantile(0.84, dim='MC').values,
        'best_deadband': ds_best['best_deadband'].values,
        'best_cutoff': ds_best['best_cutoff'].values,
    }).set_index('time')


n_resumed = n_processed = n_skipped = n_failed = 0

for day_folder in day_folders:
    day_log = _attach_day_log(day_folder.name)
    log_paths[day_folder.name] = day_log
    nb_log.info('===== %s =====', day_folder.name)
    try:
        # Resume: if this day already has bestPareto output, load it and skip run_MC.
        existing = sorted(output_folder.glob(f'{chamber_id}_{day_folder.name}*_bestPareto.nc'))
        if existing:
            summary = _build_summary(existing)
            summary_csv = output_folder / f'{chamber_id}_{day_folder.name}_summary.csv'
            if not summary_csv.exists():
                summary.to_csv(summary_csv)
            summaries[day_folder.name] = summary
            n_resumed += 1
            nb_log.info('resumed from %d existing file(s), %d timestamps', len(existing), len(summary))
            print(f'{day_folder.name}: resumed ({len(existing)} nc, {len(summary)} timestamps)')
            continue

        initializer = json_reader.Initializer(folderPath=day_folder)
        df = initializer.prepare_rawdata()

        if df.empty:
            n_skipped += 1
            nb_log.warning('skipped: empty dataframe')
            print(f'{day_folder.name}: skipped (empty dataframe)')
            continue

        nb_log.info('loaded raw: %d measurements, %d rows', df['id'].nunique(), len(df))

        processor.run_MC(
            df=df,
            chamber_id=chamber_id,
            output_folder=str(output_folder),
            sensor_precision=10,
            n_MC=4000,
            save_netcdf=True,
            log_path=day_log,
        )

        pareto_files = sorted(output_folder.glob(f'{chamber_id}_{day_folder.name}*_bestPareto.nc'))
        if not pareto_files:
            n_skipped += 1
            nb_log.warning('no bestPareto output produced')
            print(f'{day_folder.name}: no bestPareto output produced')
            continue

        summary = _build_summary(pareto_files)
        summary_csv = output_folder / f'{chamber_id}_{day_folder.name}_summary.csv'
        summary.to_csv(summary_csv)
        summaries[day_folder.name] = summary
        n_processed += 1
        nb_log.info('processed: wrote %s (%d timestamps)', summary_csv.name, len(summary))
        print(f'{day_folder.name}: processed ({len(summary)} timestamps)')

    except Exception:
        n_failed += 1
        # full traceback (including the exception type and message) goes to this
        # day's log file via nb_log.exception(). The notebook stdout only gets a
        # neutral one-liner pointing at the file — no exception text leaks here.
        nb_log.exception('failed to process %s', day_folder.name)
        print(f'{day_folder.name}: FAILED  (see {day_log.name})')

tally = f'resumed={n_resumed} processed={n_processed} skipped={n_skipped} failed={n_failed}'
nb_log.info('===== run done: %s =====', tally)
print(f'\nDone. {tally}')
print(f'Logs in: {output_folder}  (one *_processing.log per day)')

One plot per processed day folder.

In [ ]:
for day_name, summary in summaries.items():
    fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
    ax.fill_between(summary.index, summary['dcdt_q16'], summary['dcdt_q84'],
                    color='steelblue', alpha=0.3, label='16–84%')
    ax.plot(summary.index, summary['dcdt_median'], color='steelblue',
            marker='o', ms=3, lw=0.8, label='median')
    ax.set_xlabel('Time')
    ax.set_ylabel('dC/dt (HM) [ppm / s]')
    ax.set_title(f'{chamber_id} — {day_name} (MCMC, Pareto-selected)')
    ax.grid(alpha=0.3)
    ax.legend()
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()